# AIHub_KsponSpeech_Clean — EDA

## 1. Dataset Overview

- **데이터셋 이름**: AIHub KsponSpeech (Korean Spontaneous Speech) — Clean 세트
- **성격**: 읽기 음성이 아닌 **자유발화(대화체)** 한국어 음성-전사 코퍼스
- **목적**: 한국어 ASR 학습/평가용. 본 EDA는 TRAIN-ASR 프로젝트의 벤치마크 구축을 위한
  전사 규칙·특수토큰·숫자/영어 표기·음성 매핑 방식 파악이 목표
- **split 구성** (실제 `.trn` 라인 수 기준):
  | split | 발화 수 | 오디오 경로 패턴 |
  |---|---|---|
  | train | 620,000 | `KsponSpeech_01~05/KsponSpeech_XXXX/KsponSpeech_NNNNNN.pcm` |
  | dev | 2,545 | `KsponSpeech_05/.../KsponSpeech_620xxx.pcm` |
  | eval_clean | 3,000 | `KsponSpeech_eval/eval_clean/KsponSpeech_E000xx.pcm` |
  | eval_other | 3,000 | `KsponSpeech_eval/eval_other/KsponSpeech_E030xx.pcm` |
  - **합계: 628,545 발화**

> 핵심 특징(미리 보기): 오디오는 헤더 없는 **raw `.pcm`(16kHz mono 16bit)**,
> 전사는 **이중전사 `(표기형)/(발음형)`** 와 ETRI식 잡음/간투어 태그(`b/ n/ l/ o/`, `아/ 그/`)를 포함.

In [ ]:
from pathlib import Path

# 실제 서버 경로 반영
RAW_ROOT       = Path("/data/ASR/RAW/AIHub_KsponSpeech")
DATASET_NAME   = "AIHub_KsponSpeech_Clean"

# .trn 전사 파일은 scripts 폴더 안에 있음
TRANSCRIPT_DIR = RAW_ROOT / "10.한국어음성" / "KsponSpeech_scripts"
# 오디오(.pcm) 기준 디렉터리: .trn 안 경로가 KsponSpeech_01/... 로 시작하므로
# 그 폴더들이 들어있는 상위 디렉터리. (아래에서 실제 위치 확인 후 필요시 수정)
AUDIO_ROOT     = RAW_ROOT / "10.한국어음성"

TRN_FILES = {
    "train":      "train.trn",
    "dev":        "dev.trn",
    "eval_clean": "eval_clean.trn",
    "eval_other": "eval_other.trn",
}

PCM_SR, PCM_DTYPE = 16000, "int16"
SEP = "::"

print(f"[{DATASET_NAME}]")
print("TRANSCRIPT_DIR :", TRANSCRIPT_DIR)
print("AUDIO_ROOT     :", AUDIO_ROOT)
for sp, fn in TRN_FILES.items():
    p = TRANSCRIPT_DIR / fn
    print(f"  {sp:11s}: {'OK' if p.exists() else '없음'}  ({p})")

## 2. Folder Structure

`.trn` 의 `audio_path` 로부터 역으로 확인한 구조 (한 줄 = 한 발화 = 한 `.pcm` 파일):

audio (raw .pcm):
KsponSpeech_01 ~ KsponSpeech_05/ # train·dev 화자 폴더 묶음
└── KsponSpeech_XXXX/ # 세션 폴더 (4자리)
└── KsponSpeech_NNNNNN.pcm # 발화 (6자리)
KsponSpeech_eval/
├── eval_clean/KsponSpeech_E000NN.pcm
└── eval_other/KsponSpeech_E03NNN.pcm

transcript:
train.trn / dev.trn / eval_clean.trn / eval_other.trn # 각 줄 "상대오디오경로 :: 전사"

metadata:
- 별도 화자/성별 메타 파일은 .trn 만으로는 확인되지 않음 → "판단 필요"
- 화자 ID는 폴더(KsponSpeech_XXXX) 수준에서 암시됨 (명시적 speaker 컬럼 없음)


아래 셀에서 실제 디스크 구조를 직접 점검한다.

In [ ]:
# 폴더 구조 점검 (없으면 어떤 게 없는지만 보고하고 진행)
def trn_path(split):
    return TRANSCRIPT_DIR / TRN_FILES[split]

print("== .trn 파일 존재 여부 ==")
for sp in TRN_FILES:
    p = trn_path(sp)
    print(f"  {sp:11s}: {'OK' if p.exists() else '없음'}  ({p})")

print("\n== AUDIO_ROOT 하위 상위 디렉터리 ==")
if AUDIO_ROOT.exists():
    tops = sorted(d.name for d in AUDIO_ROOT.iterdir() if d.is_dir())
    print("  ", tops if tops else "(하위 디렉터리 없음)")
else:
    print("  AUDIO_ROOT 없음:", AUDIO_ROOT)

## 3. Transcript Format Analysis

- **라인 구조**: `audio_path :: transcript`  (구분자 `::`, 모든 파일에서 100% 일관)
- **발화 단위**: 한 줄 = 한 발화 = 한 `.pcm` 오디오 파일
- **발화 ID 구조**: 오디오 경로의 파일명(확장자 제외) = utterance_id
  - 예) `KsponSpeech_000001`, `KsponSpeech_620001`, `KsponSpeech_E00001`
- 아래 셀에서 총 발화 수·샘플 20개·ID 구조를 실제로 출력한다.

In [ ]:
# 전사 로더 + 포맷/ID 구조 분석
import os

def load_trn(split):
    """return list of (audio_path, transcript, utt_id)"""
    p = trn_path(split)
    rows = []
    if not p.exists():
        print(f"  [skip] {split}: 파일 없음 ({p})")
        return rows
    for line in p.open(encoding="utf-8", errors="replace"):
        line = line.rstrip("\n")
        if SEP not in line:
            continue
        apath, text = line.split(SEP, 1)
        apath, text = apath.strip(), text.strip()
        utt_id = os.path.splitext(os.path.basename(apath))[0]
        rows.append((apath, text, utt_id))
    return rows

DATA = {sp: load_trn(sp) for sp in TRN_FILES}

print("== 총 발화 수 ==")
total = 0
for sp, rows in DATA.items():
    total += len(rows)
    print(f"  {sp:11s}: {len(rows):>7,}")
print(f"  {'합계':11s}: {total:>7,}")

# 구분자 일관성
print("\n== '::' 구분자 일관성 ==")
for sp in TRN_FILES:
    p = trn_path(sp)
    if not p.exists():
        continue
    bad = sum(1 for l in p.open(encoding="utf-8", errors="replace") if SEP not in l)
    print(f"  {sp:11s}: '::' 없는 줄 {bad}")

print("\n== 샘플 20개 (train 우선) ==")
sample_src = DATA["train"] or next((r for r in DATA.values() if r), [])
for i, (a, t, u) in enumerate(sample_src[:20]):
    print(f"[{i:2d}] {u} :: {t[:70]}")

print("\n== 발화 ID 구조 (split별 첫 행) ==")
for sp, rows in DATA.items():
    if rows:
        a, t, u = rows[0]
        print(f"  {sp:11s}: id={u}  path={a}")

## 4. Special Token Analysis

대상 기호: `( )  [ ]  < >  { }  /  +  *  @  #  &`

실제 분석 결과(628,545 발화):
- `( )` 와 `/` : **이중전사**와 **잡음/간투어 태그**에서 대량 등장
- `+` (반복·말 끊김): 약 8.4만 발화
- `*` (불명확/잘림): 약 5.4만 발화
- `[ ] < > { } @ #` : **0건** (이 데이터셋엔 없음)
- `&` : 1건 (사실상 미사용)

아래 셀에서 직접 카운트한다.

In [ ]:
# 특수 기호 + ETRI식 태그 스캔
import re
ALL = [t for rows in DATA.values() for (_, t, _) in rows]
N = len(ALL); S = "\n".join(ALL)

def n_utt(pat):
    r = re.compile(pat)
    return sum(1 for t in ALL if r.search(t))

print(f"=== 일반 기호 (전체 {N:,} 발화) ===")
SYMBOLS = {
    "( ) 소괄호": r"[()]", "[ ] 대괄호": r"[\[\]]", "< > 꺾쇠": r"[<>]",
    "{ } 중괄호": r"[{}]", "/ 슬래시": r"/", "+ 플러스": r"\+",
    "* 별표": r"\*", "@": r"@", "# 샵": r"#", "& 앰퍼샌드": r"&",
}
for name, pat in SYMBOLS.items():
    c = n_utt(pat)
    print(f"  {name:11s}: {c:>8,} 발화 ({c/N*100:5.2f}%)")

print("\n=== ETRI식 잡음/간투어 태그 ===")
TAGS = {
    "b/ 숨소리":  r"(?:^|\s)b/", "n/ 잡음":  r"(?:^|\s)n/",
    "l/ 웃음":    r"(?:^|\s)l/", "o/ 타인말소리": r"(?:^|\s)o/",
    "u/ 불명확":  r"(?:^|\s)u/",
    "간투어 (아/그/어/음/저/뭐...)": r"(?:^|\s)(?:아|어|음|으|응|그|저|저기|에|뭐|좀)/",
}
for name, pat in TAGS.items():
    c = n_utt(pat)
    print(f"  {name:26s}: {c:>8,} 발화 ({c/N*100:5.2f}%)")

# 예시 출력
print("\n=== 예시 ===")
for label, pat in [("이중전사 )/(", r"\)/\("), ("+", r"\+"), ("*", r"\*"), ("b/", r"(?:^|\s)b/")]:
    ex = next((t for t in ALL if re.search(pat, t)), None)
    print(f"  [{label}] {ex[:70] if ex else '없음'}")

## 5. Number Normalization Analysis

핵심 규칙(실제 데이터): 숫자는 **거의 항상 이중전사** `(아라비아표기)/(한글발음형)` 으로 들어감.
- 아라비아 숫자를 포함한 발화: 약 **54,013개(8.6%)**, 대부분 `(...)/(...)` 내부
- 표기형엔 `123`, `2,000`, `3.5`, `70%`, 단위(`원/월/시/명/kg/cc/km`)가 붙고,
  발음형은 전부 한글(`백 이십 삼`, `이천`, `삼 점 오`, `칠십 프로`)
- 날짜/금액/퍼센트/소수/전화번호 모두 같은 패턴
  - 예) `(2018년)/(이천 십 팔 년)`, `(11,000원)/(만 천 원)`, `(70%)/(칠십 프로)`,
    `(3.5)/(삼 점 오)`, `(010)/(공 일 공)`, `(112)/(일 일 이)`

In [ ]:
# 숫자/날짜/금액/퍼센트 표기 분석 (이중전사 내부 표기형 기준)
import re
from collections import Counter

dual_pairs = []   # (표기형, 발음형)
for t in ALL:
    for m in re.finditer(r"\(([^()]*)\)/\(([^()]*)\)", t):
        dual_pairs.append((m.group(1), m.group(2)))

num_pairs = [(a, b) for (a, b) in dual_pairs if re.search(r"\d", a)]
print(f"이중전사 쌍 총: {len(dual_pairs):,} / 그중 숫자 포함 표기형: {len(num_pairs):,}")

# 아라비아 숫자 포함 발화 비율
dig_utt = sum(1 for t in ALL if re.search(r"\d", t))
print(f"아라비아 숫자 포함 발화: {dig_utt:,} ({dig_utt/N*100:.1f}%)")

def show(title, pat, k=8):
    hits = [(a, b) for (a, b) in num_pairs if re.search(pat, a)]
    print(f"\n[{title}] {len(hits):,}건")
    for a, b in hits[:k]:
        print(f"   ({a})/({b})")

show("날짜(년/월/일)", r"(년|월|일)\b|\d+/\d+")
show("시각(시/분)",   r"\d+\s*(시|분)")
show("금액(원/만/억/달러)", r"(원|만|억|달러|불|엔)")
show("퍼센트",        r"%|퍼|프로|퍼센트")
show("소수점",        r"\d+\.\d+")
show("전화/번호열",   r"^\d{2,}$")

## 6. English / Foreign Word Analysis

- 영문/약어는 대부분 **이중전사 표기형**에 라틴 그대로, 발음형은 한글로 음차
  - 예) `(KBS-X)`, `(AI)/(에이아이)`, `(USB)/(유 에스 비)`, `(PPT)/(피 피 티)`,
    `(KTX)/(케이 티 엑스)`, `(BMW)/(비엠떠블유)`, `(MRI)/(엠알아이)`
- 주의: `b/ n/ l/ o/ u/` 잡음태그가 라틴 문자라 단순 `[A-Za-z]` 검색은 과대계상됨.
  **태그를 제거하고** 측정하면 실제 영문 포함 발화는 약 **8,696개(1.38%)**.

In [ ]:
# 영어/외래어 분석 (잡음태그 b/ n/ l/ o/ u/ 제거 후 측정)
import re
from collections import Counter

tag_re = re.compile(r"(?:^|\s)[blonu]/")   # 라틴 잡음태그
def strip_tags(t): return tag_re.sub(" ", t)

eng_utt = sum(1 for t in ALL if re.search(r"[A-Za-z]", strip_tags(t)))
print(f"영문(라틴) 포함 발화 (태그 제거 후): {eng_utt:,} ({eng_utt/N*100:.2f}%)")

# 이중전사 표기형 안의 영어 토큰 빈도
eng_tokens = Counter()
for a, b in dual_pairs:                      # dual_pairs: Cell 10에서 생성
    for w in re.findall(r"[A-Za-z][A-Za-z0-9]*", a):
        eng_tokens[w.upper()] += 1
print("\n[표기형 영어 토큰 빈도 top 20]")
for w, c in eng_tokens.most_common(20):
    print(f"   {w:8s} {c}")

print("\n[영어 이중전사 예시]")
shown = 0
for a, b in dual_pairs:
    if re.search(r"[A-Za-z]", a):
        print(f"   ({a})/({b})"); shown += 1
    if shown >= 12:
        break

## 7. Dual Transcription Analysis

- 패턴: **`(표기형)/(발음형)`** — KsponSpeech 핵심 표기 규칙
- 실제 발견: `)/(` 연결부 **90,876건** (숫자/영어/단위/날짜/금액 등 광범위)
- 사용 규칙(데이터 기준):
  1. 숫자·영문·기호 등 **소리와 표기가 다른 토큰**을 `(원형)/(한글 발음)` 로 병기
  2. 왼쪽 = 원형(아라비아/라틴/기호 포함), 오른쪽 = **순수 한글 발음형**
  3. 학습/평가 시 **둘 중 하나를 골라야 함** (표기형 vs 발음형 정규화 결정 필요)
- 예) `(서울역)`류 순수 한글-한글 쌍은 드물고, 대부분 숫자/영어가 좌측

In [ ]:
# 이중전사 (표기형)/(발음형) 정량 + 분류
import re
junction = S.count(")/(")
print(f")/(  연결부 총: {junction:,}")
print(f"완성된 (..)/(.. ) 쌍: {len(dual_pairs):,}")   # Cell 10에서 추출

# 좌측(표기형) 유형 분류
cls = {"숫자포함": 0, "영문포함": 0, "한글전용": 0, "기타": 0}
for a, b in dual_pairs:
    if re.search(r"\d", a):            cls["숫자포함"] += 1
    elif re.search(r"[A-Za-z]", a):    cls["영문포함"] += 1
    elif re.fullmatch(r"[가-힣\s.,]+", a): cls["한글전용"] += 1
    else:                              cls["기타"] += 1
print("\n[표기형 유형 분포]")
for k, v in cls.items():
    print(f"   {k:8s}: {v:>7,} ({v/max(len(dual_pairs),1)*100:5.1f}%)")

print("\n[예시 12개]")
for a, b in dual_pairs[:12]:
    print(f"   ({a})/({b})")

In [ ]:
import re

hangul_only, etc = [], []
for a, b in dual_pairs:
    if re.search(r"\d", a):           continue
    if re.search(r"[A-Za-z]", a):     continue
    if re.fullmatch(r"[가-힣\s.,]+", a): hangul_only.append((a, b))
    else:                             etc.append((a, b))

print(f"한글전용: {len(hangul_only):,}개 — 예시 30개")
for a, b in hangul_only[:30]:
    print(f"   ({a})/({b})")

print(f"\n기타: {len(etc)}개 — 전부")
for a, b in etc:
    # 어떤 문자가 '기타'로 분류됐는지 같이 표시
    others = sorted(set(re.findall(r"[^가-힣\s.,A-Za-z0-9]", a)))
    print(f"   ({a})/({b})   ← 특이문자:{others}")

## 8. Token Type Classification

| 토큰 | 의미 | 분류 | 근거(데이터) |
|---|---|---|---|
| `b/` | 숨소리(breath) | **음향정보** | ETRI식 잡음태그, 약 235k 발화 |
| `l/` | 웃음(laugh) | **음향정보** | 약 38k 발화 |
| `n/` | 주변잡음(noise) | **음향정보** | 약 58k 발화 |
| `o/` | 타인 말소리 | **음향정보** | 약 154k 발화 |
| `아/ 그/ 어/ 음/ 저/ 뭐/ …` | 간투어(filler) | **전사 규칙** | 약 126k 발화 |
| `+` | 반복·말 끊김 | **전사 규칙** | 약 84k 발화 |
| `*` | 불명확/잘린 발성 | **전사 규칙** | 약 54k 발화 |
| `u/` | 완전 불명확 | **전사 규칙** | 약 10k 발화 |
| `(표기형)/(발음형)` | 이중전사 | **전사 규칙** | 90,876건 |
| 이름/전화/주소 마스킹 | 비식별화 | **해당 없음** | 마스킹 토큰 미발견 |
| `[] <> {} @ #` | — | **해당 없음** | 0건 |
| `&` | — | **판단 필요** | 1건뿐, 용도 불명 |

> 비식별화(PII) 전용 토큰은 발견되지 않음. 위 분류는 실제 등장 토큰만 반영.

In [ ]:
# 분류표를 실제 카운트로 재확인 (표가 데이터와 맞는지 검증용)
import re
def cnt(pat): return sum(1 for t in ALL if re.search(pat, t))
checks = {
    "음향 b/": r"(?:^|\s)b/", "음향 l/": r"(?:^|\s)l/",
    "음향 n/": r"(?:^|\s)n/", "음향 o/": r"(?:^|\s)o/",
    "규칙 간투어": r"(?:^|\s)(?:아|어|음|으|응|그|저|저기|에|뭐|좀)/",
    "규칙 +": r"\+", "규칙 *": r"\*", "규칙 u/": r"(?:^|\s)u/",
    "규칙 이중전사": r"\)/\(",
    "PII 마스킹 후보([NAME]등)": r"\[[A-Z가-힣]+\]|@@@|＊＊＊",
}
print("== 분류 검증 카운트 ==")
for k, p in checks.items():
    print(f"  {k:22s}: {cnt(p):>8,}")

## 8-1. 비식별화 토큰 확인

In [ ]:
import re
from collections import Counter, defaultdict

# ALL: Cell 8에서 만든 전체 전사 리스트
acoustic_o = re.compile(r"(?:^|\s)o/")   # 음향 태그 o/ 는 제외

# 각 문자별 '연속 런'을 찾는다  예) ooo, OOOO, ㅇㅇ
patterns = {
    "ㅇ (한글 이응)": re.compile(r"ㅇ+"),
    "o (라틴 소문자)": re.compile(r"(?<![A-Za-z])o+(?![A-Za-z])"),
    "O (라틴 대문자)": re.compile(r"(?<![A-Za-z])O+(?![A-Za-z])"),
}

for name, pat in patterns.items():
    length_count = Counter()         # 런 길이 -> 등장 횟수
    examples = defaultdict(list)     # 런 길이 -> 예시 발화
    utt_with = 0
    for t in ALL:
        t_clean = acoustic_o.sub(" ", t)
        runs = pat.findall(t_clean)
        if runs:
            utt_with += 1
            for r in runs:
                length_count[len(r)] += 1
                if len(examples[len(r)]) < 3:
                    examples[len(r)].append(t)

    print(f"===== {name} =====")
    print(f"  포함 발화 수: {utt_with:,}")
    if not length_count:
        print("  (없음)\n")
        continue
    print("  [개수(런 길이)별 등장 횟수]")
    for L in sorted(length_count):
        token = name[0] * L
        print(f"    {token:>8s} ({L}글자): {length_count[L]:>6,}회")
        for ex in examples[L][:2]:
            print(f"          예) {ex[:70]}")
    print()

In [ ]:
import re, numpy as np
from IPython.display import display, Audio, HTML

acoustic_o = re.compile(r"(?:^|\s)o/")
o_low  = re.compile(r"(?<![A-Za-z])o+(?![A-Za-z])")
o_up   = re.compile(r"(?<![A-Za-z])O+(?![A-Za-z])")

# DATA: Cell 6에서 만든 {split: [(apath, text, utt_id), ...]}
hits = []   # (split, utt_id, apath, text, 종류)
for split, rows in DATA.items():
    for apath, text, uid in rows:
        clean = acoustic_o.sub(" ", text)
        kinds = []
        if o_low.search(clean): kinds.append("o")
        if o_up.search(clean):  kinds.append("O")
        if kinds:
            hits.append((split, uid, apath, text, "+".join(kinds)))

print(f"o/O 후보 발화: {len(hits)}개 (자동재생 없음)\n")

def load_pcm(p): return np.fromfile(p, dtype=PCM_DTYPE)

for i, (split, uid, apath, text, kind) in enumerate(hits):
    full = AUDIO_ROOT / apath
    # 해당 글자에 형광표시
    marked = re.sub(r"(?<![A-Za-z])([oO]+)(?![A-Za-z])",
                    r"<mark>\1</mark>", acoustic_o.sub(" ", text))
    display(HTML(
        f"<b>[{i:02d}]</b> <code>{uid}</code> · {split} · 종류:{kind}<br>"
        f"<small>{apath} · {'OK' if full.exists() else '오디오없음'}</small><br>{marked}"
    ))
    if full.exists():
        display(Audio(load_pcm(full), rate=PCM_SR, autoplay=False))
    display(HTML("<hr>"))

In [ ]:
import re
from collections import Counter, defaultdict

# DATA: Cell 6의 {split: [(apath, text, utt_id), ...]}
acoustic_o = re.compile(r"(?:^|\s)o/")

# 단어 안(예: 4DX, TX, XS)이 아닌 '독립 런'만: 앞뒤가 영문이 아닐 때
x_low = re.compile(r"(?<![A-Za-z])x+(?![A-Za-z])")
x_up  = re.compile(r"(?<![A-Za-z])X+(?![A-Za-z])")

for label, pat in [("x (소문자)", x_low), ("X (대문자)", x_up)]:
    length_count = Counter()
    examples = defaultdict(list)
    utt_with = 0
    for split, rows in DATA.items():
        for apath, text, uid in rows:
            clean = acoustic_o.sub(" ", text)
            runs = pat.findall(clean)
            if runs:
                utt_with += 1
                for r in runs:
                    length_count[len(r)] += 1
                    if len(examples[len(r)]) < 3:
                        examples[len(r)].append((split, uid, text))
    print(f"===== {label} =====")
    print(f"  포함 발화 수: {utt_with:,}")
    if not length_count:
        print("  (없음)\n"); continue
    print("  [개수(런 길이)별 등장 횟수]")
    for L in sorted(length_count):
        token = label[0] * L
        print(f"    {token:>8s} ({L}글자): {length_count[L]:>6,}회")
        for sp, uid, ex in examples[L][:2]:
            print(f"          예) [{sp}/{uid}] {ex[:65]}")
    print()

In [ ]:
import re, numpy as np
from IPython.display import display, Audio, HTML

xpat = re.compile(r"[xX]")

# DATA: Cell 6의 {split: [(apath, text, utt_id), ...]}
hits = []
for split, rows in DATA.items():
    for apath, text, uid in rows:
        if xpat.search(text):
            hits.append((split, uid, apath, text))

print(f"x/X 포함 발화: {len(hits)}개 (자동재생 없음)\n")

def load_pcm(p): return np.fromfile(p, dtype=PCM_DTYPE)

for i, (split, uid, apath, text) in enumerate(hits):
    full = AUDIO_ROOT / apath
    marked = xpat.sub(lambda m: f"<mark>{m.group()}</mark>", text)
    display(HTML(
        f"<b>[{i:03d}]</b> <code>{uid}</code> · {split}<br>"
        f"<small>{apath} · {'OK' if full.exists() else '오디오없음'}</small><br>{marked}"
    ))
    if full.exists():
        display(Audio(load_pcm(full), rate=PCM_SR, autoplay=False))
    display(HTML("<hr>"))

In [ ]:
import re, numpy as np
from IPython.display import display, Audio, HTML
from collections import defaultdict

TAG_LETTERS = ["b", "n", "l", "o", "u"]   # 슬래시 없는 잡음 태그 후보 (소문자)
MAX_PLAY = 200                              # 총 재생 위젯 수 상한

# 단어 안이 아니고, 바로 뒤가 슬래시도 아닌 '맨 글자' 만
bare = {c: re.compile(rf"(?<![A-Za-z]){c}(?!/)(?![A-Za-z])") for c in TAG_LETTERS}

found = defaultdict(list)   # letter -> [(split, uid, apath, text)]
for split, rows in DATA.items():
    for apath, text, uid in rows:
        for c in TAG_LETTERS:
            if bare[c].search(text):
                found[c].append((split, uid, apath, text))

print("=== 슬래시 없는 잡음 태그 글자별 발화 수 ===")
for c in TAG_LETTERS:
    print(f"  '{c}' : {len(found[c]):>5,} 발화")
total = sum(len(v) for v in found.values())
print(f"  합계(중복 가능): {total:,}\n")

def load_pcm(p): return np.fromfile(p, dtype=PCM_DTYPE)

played = 0
for c in TAG_LETTERS:
    items = found[c]
    if not items:
        continue
    display(HTML(f"<h3>'{c}' (슬래시 없음) — {len(items)}개</h3>"))
    for split, uid, apath, text in items:
        if played >= MAX_PLAY:
            display(HTML(f"<b>… MAX_PLAY({MAX_PLAY}) 도달, 나머지 생략</b>"))
            break
        full = AUDIO_ROOT / apath
        marked = bare[c].sub(lambda m: f"<mark>{m.group()}</mark>", text)
        display(HTML(
            f"<code>{uid}</code> · {split}<br>"
            f"<small>{apath} · {'OK' if full.exists() else '오디오없음'}</small><br>{marked}"
        ))
        if full.exists():
            display(Audio(load_pcm(full), rate=PCM_SR, autoplay=False))
        display(HTML("<hr>"))
        played += 1
    if played >= MAX_PLAY:
        break

## 9. Audio Mapping Validation

각 `.trn` 의 `audio_path` 가 `AUDIO_ROOT` 아래 실제 `.pcm` 로 존재하는지 매칭 점검.
출력: split별 전체/매칭/누락 수 + 누락 샘플 일부.
(오디오가 아직 서버에 없으면 전부 누락으로 보고됨 — 경로만 확인하는 용도로도 사용)

In [ ]:
# 전사 ↔ .pcm 매핑 검증
def validate(split, max_show=5):
    rows = DATA[split]
    if not rows:
        print(f"  [skip] {split}: 전사 없음"); return
    miss = []
    for apath, _, _ in rows:
        if not (AUDIO_ROOT / apath).exists():
            miss.append(apath)
    matched = len(rows) - len(miss)
    print(f"  {split:11s}: 전체 {len(rows):>7,} | 매칭 {matched:>7,} | 누락 {len(miss):>7,}")
    for m in miss[:max_show]:
        print(f"        누락 예) {m}")

print("== Audio Mapping Validation ==")
for sp in TRN_FILES:
    validate(sp)

## 10. Audio Review Cell

- seed 고정 후 100개 무작위 추출 → index / utterance_id / audio_path / transcript / duration 표시
- `.pcm` 은 헤더 없는 raw 라 `soundfile` 로 못 읽음 → **`numpy.fromfile(int16)` + `Audio(rate=16000)`**
- **자동재생 안 함** (`autoplay=False`), 각 항목을 `display` 로 나열해 클릭 재생

In [ ]:
# 100개 샘플 청취 (.pcm raw 16kHz mono 16bit)
import random, numpy as np
from IPython.display import display, Audio, HTML

SEED = 42
N_SAMPLES = 100
REVIEW_SPLIT = "train"      # 바꿔서 dev/eval_clean/eval_other 검토 가능

rows = DATA[REVIEW_SPLIT]
random.seed(SEED)
picks = random.sample(rows, min(N_SAMPLES, len(rows)))

def pcm_duration_sec(path):
    try:
        return path.stat().st_size / 2 / PCM_SR   # 16bit=2byte
    except OSError:
        return None

def load_pcm(path):
    data = np.fromfile(path, dtype=PCM_DTYPE)     # raw little-endian int16
    return data

print(f"[{REVIEW_SPLIT}] seed={SEED}, {len(picks)}개 (자동재생 없음)\n")
for idx, (apath, text, uid) in enumerate(picks):
    full = AUDIO_ROOT / apath
    dur = pcm_duration_sec(full)
    dur_s = f"{dur:.2f}s" if dur is not None else "N/A(파일없음)"
    display(HTML(
        f"<b>[{idx:03d}]</b> <code>{uid}</code> · {dur_s}<br>"
        f"<small>{apath}</small><br>{text}"
    ))
    if full.exists():
        display(Audio(load_pcm(full), rate=PCM_SR, autoplay=False))
    display(HTML("<hr>"))

## 11. Findings / Issues / TODO

### 확인된 특징 (실제 데이터 근거)
- **포맷**: `audio_path :: transcript`, 한 줄 = 한 발화 = 한 `.pcm` (628,545 발화)
- **숫자 규칙**: 거의 항상 이중전사 `(아라비아)/(한글발음)`, 숫자 포함 발화 8.6%
- **영어 규칙**: 영문/약어는 표기형에 라틴 그대로, 발음형은 한글 음차 (태그 제외 시 1.38%)
- **이중전사**: `(표기형)/(발음형)` 90,876건 — 이 데이터셋의 핵심 규칙
- **특수토큰**: 음향(`b/ n/ l/ o/`)·전사규칙(`아/ 그/`, `+`, `*`, `u/`)만 존재.
  `[] <> {} @ #` 0건, PII 마스킹 토큰 미발견
- **오디오**: 헤더 없는 raw `.pcm` (16kHz mono 16bit) — soundfile 불가, numpy 직접 로드 필요

### 벤치마크 구축 시 고려사항
- **text normalization 필수**: 이중전사 중 한쪽 선택 규칙을 먼저 정해야 함
  - 표기형(`(2018년)` 쪽) vs 발음형(`(이천 십 팔 년)` 쪽) — 평가 지표(CER/WER)에 큰 영향
- **특수토큰 제거 여부**: `b/ n/ l/ o/ u/ + *` 및 간투어 태그를 학습/평가 전 제거할지 결정
- **발음형 사용 여부**: 음향모델 일관성엔 발음형이 유리할 수 있으나, 표기 복원이 필요한 태스크면 표기형 보존
- **speaker 정보**: .trn 에 speaker 컬럼 없음 → 화자 단위 분리/평가가 필요하면 별도 메타 확보 필요

### 추가 확인 필요
- **판단 어려운 토큰**: `&`(1건) 용도, 일부 한글-한글 이중전사의 정의
- **음성 품질**: eval_clean vs eval_other 잡음 태그(`n/ o/`) 분포 차이 정량화
- **매핑 이슈**: Audio Mapping Validation 의 누락 건 원인(.pcm 미배치 / 경로 규칙 차이) 점검